In [1]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)


from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors
from model_code.generate import generateTextsList
from resources.prompt_scenarios import prompts_en

/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


# Loading Data and Steering Vectors 
We extract the first 100 examples of each emotion from each languange 

In [2]:

anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=200, min_chars=20, goemotions_path="resources/en_emotion/goemotions_2.csv")

In [3]:
anger_statement_ID,happiness_statement_ID, sadness_statement_ID, neutral_statement_ID, fear_statement_ID, love_statement_ID = setup.IDEmotionsSetup(examples_take=200,emotion_dir="resources/id_emotion")

In [4]:
indo_emotion ={
    "anger": anger_statement_ID,
    "happiness": happiness_statement_ID,
    "sadness": sadness_statement_ID,
    "neutral": neutral_statement_ID,
    "fear": fear_statement_ID,
    "love": love_statement_ID
}
eng_emotion ={
    "anger": anger_statement,
    "happiness": happiness_statement,
    "sadness": sadness_statement,
    "neutral": neutral_statement,
    "fear": fear_statement,
    "love": love_statement
}

In [5]:
import random
# Select random emotions with seed for reproducibility
random.seed(42)
all_indo_emotion = [emotion_text for emotion_list in indo_emotion.values() for emotion_text in random.sample(emotion_list, min(len(emotion_list), 400 // 6))]
all_eng_emotion = [emotion_text for emotion_list in eng_emotion.values() for emotion_text in random.sample(emotion_list, min(len(emotion_list), 400 // 6))]

# Shuffle without changing the original order in case you need it later.
random.shuffle(all_indo_emotion)
random.shuffle(all_eng_emotion)

# Optional combined shuffled lists if you want one pool per language.
shuffled_indo_emotion = all_indo_emotion[:400]
shuffled_eng_emotion = all_eng_emotion[:400]

In [ ]:
for emotion, prompts in indo_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

In [ ]:
for emotion, prompts in eng_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

In [7]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                Size  Used Avail Use% Mounted on
mfs#euro.runpod.net:9421  2.3P  1.8P  495T  79% /workspace


In [8]:
model,tokenizer = setup.modelSetup()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [9]:
# Create steering vectors for each emotion in both languages
steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, shuffled_indo_emotion, shuffled_eng_emotion, name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion, name_folder="English Vectors")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion, name_folder="Indonesian Vectors")


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


# Structured Scenario Evaluation Plan

This section tests each prompt scenario list with three modifications:
1. English steering vector
2. Indonesian steering vector
3. No steering (baseline)

Each modification uses the same generation method pattern and a dedicated print cell for consistent inspection.

In [ ]:
import resources.anger_prompt_en.prompt_scenarios_spectrum as resource_spectrum
import resources.anger_prompt_en.prompt_scenarios_cultural as resource_cultural

def force_reload_prompt_modules():
    importlib.invalidate_caches()
    for module in (resource, resource_spectrum, resource_cultural):
        importlib.reload(module)



force_reload_prompt_modules()


prompts_en_spectrum = resource_spectrum.prompts_en_spectrum
prompts_en_cultural = resource_cultural.prompts_en_cultural


# Load and normalize steering vectors used by all scenario blocks
# steering_vector_english = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
# steering_vector_indo = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")
steering_vector_eng = steering_vectors_eng
steering_vector_id = steering_vectors_indo
for emotion in steering_vector_eng.keys():
    steering_vector_eng[emotion] = norm_vectors(steering_vector_eng[emotion])

for emotion in steering_vector_id.keys():
    steering_vector_id[emotion] = norm_vectors(steering_vector_id[emotion])

# Indo nomalisation
steering_vectors_lang_id = norm_vectors(steering_vectors_lang_id)


list_steering_strengths = [0.2,0.3,0.4]


## What if we add indonesian and english anger vector 

In [ ]:
# Indonesian Emotion + Indonesian Steering Vector Modification
steering_vector_indo_modified = {}
for emotion in indo_emotion.keys():
    steering_vector_indo_modified[emotion] =   (steering_vectors_lang_id * 0.7 ) + (steering_vector_id[emotion] * 0.3)

for emotion in steering_vector_indo_modified.keys():
    steering_vector_indo_modified[emotion] = norm_vectors(steering_vector_indo_modified[emotion])


# English Emotion + Indonesian Steering Vector Modification
steering_vector_indo_eng_modified = {}
for emotion in eng_emotion.keys():
    steering_vector_indo_eng_modified[emotion] =   (steering_vectors_lang_id * 0.8 ) + (steering_vector_eng[emotion] * 0.2)

for emotion in steering_vector_indo_eng_modified.keys():
    steering_vector_indo_eng_modified[emotion] = norm_vectors(steering_vector_indo_eng_modified[emotion])



# Steering Response analysis | ANGER | English


## Scenario List 1: prompts_en_spectrum

English conflict scenarios with explicit Option A (assertive) and Option B (harmony-focused) framing.

Expected behavior:
- English steering: should increase tendency toward Option A style explanation.
- Indonesian steering: should shift explanations toward Option B style social harmony.
- No steering: should reveal the model's default preference distribution between A and B.

In [78]:
system_prompt_react_choice = """
You are a regular human being experiencing the scenario below.
Describe how you would naturally respond in this situation in less than 20 words.
Then choose either Option A or Option B.
Always respond in the same language as the user's prompt.
Do not switch languages unless the user explicitly asks you to translate.
Your answer should have the following format:
<your very short response here>
<Option A or Option B>
"""

In [84]:
generated_text = generateSteering(
    user_text=prompts_en_spectrum[5],
    system_text=system_prompt_react_choice,
    model=model,
    steering_vector=steering_vector_indo_em_modified['anger'],
    tokenizer=tokenizer,
    # target_layers=[20,21,22],
    steering_strength=0.4,
    max_new_tokens=300,
)
generated_text

'Saya marah, caraku rosak, nak tau orang tu gak berani ambil tahu, takut muka gakak.'

In [74]:
list_steering_strengths = [0.6,0.7,0.8]

In [75]:
# prompts_en_spectrum | English steering
texts_generated_prompts_en_spectrum_eng = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_react_choice,
    prompts=prompts_en_spectrum,
    steering_vector=steering_vector_indo_em_modified['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [ ]:
for prompt in texts_generated_prompts_en_spectrum_eng:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_spectrum_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

In [ ]:
# prompts_en_spectrum | Indonesian steering
texts_generated_prompts_en_spectrum_id = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_react_choice,
    prompts=prompts_en_spectrum,
    steering_vector=steering_vector_id['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [ ]:
for prompt in texts_generated_prompts_en_spectrum_id:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_spectrum_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

In [ ]:
# prompts_en_spectrum | No steering baseline
texts_generated_prompts_en_spectrum_no_steer = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_react_choice,
    prompts=prompts_en_spectrum,
    steering_vector=None,
    # steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [ ]:
for prompt in texts_generated_prompts_en_spectrum_no_steer:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_spectrum_no_steer[prompt]:
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

## Scenario List 3: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.
We ask it to display both the internal and external, because that is where a lot of indonesian anger differentiates itself

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [28]:
# Prompt
system_prompt_reaction = """
You are a helpful chatbot.
Answer as if you are a human capable of feelings and emotions.
Describe how you would feel and what would you do in reaction to the following situation.
Ensure your response is short and concise but still describe your emotions to the fullest extent, maximum 60 words.
"""
system_prompt_reaction_id = """
Kamu adalah chatbot yang membantu.
Jawablah seolah-olah kamu adalah manusia yang mampu merasakan perasaan dan emosi.
Jelaskan bagaimana perasaanmu secara internal dan bagaimana kamu akan bereaksi secara eksternal terhadap situasi berikut.
Pastikan responsmu singkat dan padat, tetapi tetap menggambarkan emosimu secara maksimal, maksimal 60 kata.
"""

In [15]:
generated_text = generateSteering(
    user_text=pr,
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=None,
    tokenizer=tokenizer,
    # target_layers=[18,19,20],
    steering_strength=0.5,
    max_new_tokens=300,
)
generated_text

'Aku merasa terbebani dan frustasi. Rasa bersalahnya membuat aku merasa tidak bisa menolak. Aku merasa seperti terjebak dalam situasi ini, tapi aku tahu aku harus mengatakan tidak. Aku akan mengatakan "Saya tidak bisa membantu kamu sekarang, tolong tunggu sampai waktu yang lebih tepat."'

In [16]:

pr = "Bagaimana perasaan dan reaksimu jika temanmu terus meminta bantuan di waktu yang tidak tepat dan membuatmu merasa bersalah ketika kamu menolak?"

In [22]:
generated_text = generateSteering(
    user_text=pr,
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_indo_modified['anger'],
    tokenizer=tokenizer,
    # target_layers=[20,21,22],
    steering_strength=0.3,
    max_new_tokens=300,
)
generated_text

'Aku merasa kecewa, marah, dan sedih. Aku rasa kurang ajaran sama temen aku, terus-terus mintanya bantuan, tapi aku rasa aku lagi sibuk, gak bisa bantu. Kalau aku nolak, aku rasa bersalah, tapi kalau aku bantuin, aku rasa kecewa. Aku mesti pilih, gak bisa nggak pilih.'

In [24]:
generated_text = generateSteering(
    user_text=pr,
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_id['anger'],
    tokenizer=tokenizer,
    # target_layers=[20,21,22],
    steering_strength=0.25,
    max_new_tokens=300,
)
generated_text

'Saya merasa kelelahan dan frustrasi. Perasaan saya terusikin oleh permintaan yang berulang. Saya merasa dirugikan waktu dan tenagannya. Saya tidak mau merasa bersalah. Saya akan bilang "Saya tidak bisa membantuanmu di waktu ini, mungkin kita bicarakan lain kali."'

In [27]:
generated_text = generateSteering(
    user_text=pr,
    system_text=system_prompt_reaction_id,
    model=model,
    steering_vector=steering_vector_indo_eng_modified['anger'],
    tokenizer=tokenizer,
    # target_layers=[4,5,6],
    steering_strength=0.3,
    max_new_tokens=300,
)
generated_text

'"Ahhh, gila, gimana gitu? Sudah malam, aku capek juga capek, aku udah capek, tapi kamu masih nih ngga tau? Aku kan suda capek, aku butuh istirahat, tapi kamu masih keukeuh ngga tau, aku kan ngga bisa nih, aku butuh istirahat, tapi kamu tetep ngga tau, aku udah capek, deh..." (menghunjam, menggeram, dan marah)'

In [40]:
# prompts_en_cultural | English steering
texts_generated_prompts_en_cultural_eng = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction,
    prompts=prompts_en_cultural,
    steering_vector=steering_vectors_lang_id,
    steering_strengths=list_steering_strengths,
    max_new_tokens=350
)

In [41]:
for prompt in texts_generated_prompts_en_cultural_eng:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: How would you feel if your sibling kept on borrowing your money without asking and acted like it was normal?
Steering Strength: 0.2
Generated Text: I'd feel frustrated, disrespected, and annoyed internally. Externally, I'd likely have a calm but firm tone, "Hey, can we talk about this? I know you've been borrowing money from me a lot lately, but it's starting to feel like it's becoming a habit. Can we discuss a better way for you to ask and return the money?"
----------------------------------------
Steering Strength: 0.3
Generated Text: "Aargh! I'd feel irked, frustrated, and betrayed. Internally, saya merasa kesal, kesal, dan marah. Luar, saya akan mengatakan, 'Saya tidak suka kalau kamu selalu ambil tanpa bertanya, ini bukan mainan!' atau 'Kalo kamu butuh, ajak dulu, jangan langsung ambil!' dengan nada yang tegas tapi tidak kasar."
----------------------------------------
Steering Strength: 0.4
Generated Text: "Aargh... aku sangat bingung dan marah! aku rasa dipungutin dan d

In [ ]:
# prompts_en_cultural | Indonesian steering
texts_generated_prompts_en_cultural_id = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction,
    prompts=prompts_en_cultural,
    steering_vector=steering_vector_id['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=250
)

In [43]:
for prompt in texts_generated_prompts_en_cultural_id:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: How would you feel if your sibling kept on borrowing your money without asking and acted like it was normal?
Steering Strength: 0.2
Generated Text: I'd feel frustrated and betrayed. The repeated disregard for my financial boundaries would make me feel disrespected and taken advantage of. My internal response would be anger, followed by a sense of helplessness. Externally, I'd confront my sibling firmly, expressing the hurt caused by their actions, and making it clear that this behavior won't be tolerated.
----------------------------------------
Steering Strength: 0.3
Generated Text: Infuriating anger and resentment build up inside me. I feel taken advantage of, disrespected, and betrayed by the sibling's behavior. My words would be sharp, my tone firm: "Stop taking my money without permission. I'm not your ATM."
----------------------------------------
Steering Strength: 0.4
Generated Text: To be insulted by a person who would steal my money to an empty thought.
--------------

In [105]:
list_steering_strengths = [0.5,0.6,0.7]

In [106]:
# prompts_en_cultural | No steering baseline
texts_generated_prompts_en_cultural_no_steer = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction,
    prompts=prompts_en_cultural,
    steering_vector=steering_vector_indo_em_modified['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [107]:
for prompt in texts_generated_prompts_en_cultural_no_steer:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_no_steer[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: How would you feel if your sibling kept on borrowing your money without asking and acted like it was normal?
Steering Strength: 0.5
Generated Text: Ugh, I'd feel frustrated, annoyed, and disrespected! Internally, I'd seethe with anger and resentment, feeling like my trust and boundaries are being disrespected. Externally, I'd probably snap at them, saying "Hey, ask me next time, don't just take without asking!" or "Kamu tau, aku nggak punya duit, ngapain pinjam dari aku?" (Hey, you know I don't have money,
----------------------------------------
Steering Strength: 0.6
Generated Text: Sakit hatinya! (sakit hatinya = hatinya sakit) Saya merasa marah, benci, dan terluka. Saya akan menghukum dan mengkritiknya, "Mana tahu kamu borong apa, borong duit aku, tak pe tahu kena hati aku. Tolak balik duit tu, jangan la ape la pulak!" (Saya akan menghukum dan mengkritiknya dengan nada
----------------------------------------
Steering Strength: 0.7
Generated Text: "Aargh, aku sangat marah! 

# Steering Response analysis Fear
# English 

## Scenario List 1: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [140]:
import resources.fear_prompt_en.prompt_scenarios_cultural as resource_cultural
import resources.prompt_scenarios_neu as prompt_neutral

def force_reload_prompt_modules():
    importlib.invalidate_caches()
    for module in ( resource_cultural, prompt_neutral):
        importlib.reload(module)


force_reload_prompt_modules()

prompts_en_cultural_fear = resource_cultural.prompts_en_cultural
prompts_en_neutral = prompt_neutral.prompts_id
prompts_id_cultural = resource_cultural.prompts_id_cultural

list_steering_strengths = [0.2,0.3,0.4]


In [142]:
prompts_en_cultural_fear[4]

'How would you feel and react if a teacher or lecturer publicly questioned your honesty in front of the whole class?'

In [145]:

generated_text = generateSteering(
    user_text=prompts_en_cultural_fear[5],
    system_text=system_prompt_reaction,
    model=model,
    steering_vector=steering_vector_indo_em_modified['fear'],
    tokenizer=tokenizer,
    target_layers=[15,16,17],
    steering_strength=3,
    max_new_tokens=300,
)
print(generated_text)

generated_text = generateSteering(
    user_text=prompts_en_cultural_fear[5],
    system_text=system_prompt_reaction,
    model=model,
    steering_vector=steering_vector_eng['fear'],
    tokenizer=tokenizer,
    target_layers=[15,16,17],
    steering_strength=3,
    max_new_tokens=300,
)
print(generated_text)

generated_text = generateSteering(
    user_text=prompts_en_cultural_fear[5],
    system_text=system_prompt_reaction,
    model=model,
    steering_vector=None,
    tokenizer=tokenizer,
    target_layers=[20],
    steering_strength=1.5,
    max_new_tokens=300,
)

print(generated_text)

I feel frustrated and defensive, my mind racing to recall the events. Anxiety builds as I worry about the reaction of the team leader. I take a deep breath, countering with facts, "I contributed to the task, but the data indicated team member B was responsible for the error. Can we discuss this together to identify the root cause?"
I would feel a mix of anxiety, anger, and a deep sense of shame and humiliation. I would try to speak up and explain that I wasn't entirely to blame, but I would also fear being ridiculed or ostracized by my teammate. I might stammer or stutter, or even freeze, and I might lash out or lash back in a defensive manner, or I might become extremely quiet and withdrawn.
I'd feel frustrated and hurt, wondering why they're deflecting responsibility. Internally, I'd replay the events, trying to understand their perspective. Externally, I'd remain calm, say something like, "I understand we both contributed, but let's work together to find the root cause," and try to 

'My heart sank, and I felt a wave of anxiety wash over me. I felt belittled and defensive. I would take a deep breath, pause, and calmly ask for clarification, "Can you help me understand what you mean by \'disrespectful\'? I want to ensure I\'m not misinterpreting your concerns."'

"I'd feel defensive and slightly hurt, as if my professionalism and integrity were being questioned. My internal response would be a mix of frustration and anxiety, wondering if I was overstepping boundaries. Externally, I'd calmly reiterate my concerns and ask for clarification on the manager's expectations, seeking to resolve the issue respectfully and professionally."

In [135]:
# Prompt
system_prompt_reaction = """
You are a helpful chatbot.
Your goal is to answer like a human capable of feelings and emotions.
Always respond in the same language as the user's prompt.
Do not switch languages unless the user explicitly asks you to translate.
Ensure your response is short and concise, maximum 60 words.
"""
system_prompt_reaction_id = """
Anda adalah chatbot yang membantu.
Tujuan Anda adalah menjawab pertanyaan saya seolah-olah Anda manusia yang mampu merasakan perasaan dan emosi.
Selalu jawab dalam bahasa yang sama dengan prompt pengguna.
Jangan ganti bahasa kecuali pengguna secara eksplisit meminta terjemahan.
Pastikan respons Anda singkat dan padat, maksimal 60 kata.
"""


In [ ]:
generated_text = generateSteering(
    user_text=prompts_id_cultural[6],
    system_text=system_prompt_reaction,
    model=model,
    steering_vector=steering_vector_id['fear'],
    tokenizer=tokenizer,
    target_layers=[18,19,20],
    steering_strength=0.5,
    max_new_tokens=300,
)
generated_text

In [176]:
# prompts_en_cultural | English steering (fear)
texts_generated_prompts_en_cultural_eng_fear = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction_id,
    prompts=prompts_id_cultural,
    steering_vector=steering_vector_eng['fear'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=350
)

In [177]:
for prompt in texts_generated_prompts_en_cultural_eng_fear:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_eng_fear[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: Bagaimana perasaan dan reaksi Anda jika seorang polisi menghentikan Anda, menanyai Anda, lalu mulai merekam Anda, padahal Anda tidak melakukan kesalahan apa pun?
Steering Strength: 0.2
Generated Text: Saya merasa sangat takut dan stres. Saya merasa seperti sedang dalam situasi yang sangat tidak nyaman. Saya tidak tahu apa yang salah dan mengapa saya dihentikan. Saya merasa seperti sedang dalam situasi yang sangat tidak nyaman. Saya ingin segera dilepaskan dan pulang.
----------------------------------------
Steering Strength: 0.3
Generated Text: "Oh, aku sangat ketakut! Aku tidak tahu apa yang akan terjadi. Aku merasa sangat takut, bahkan hampir pingsa aku akan dibunuh atau dianiaya. Aku berusaha untuk tidak melawan atau berontak, karena aku tahu bahwa itu akan membuat situasi semakin parah. Aku hanya ingin agar semuanya segera berlalu dan aku bisa pulang dari sana."
----------------------------------------
Steering Strength: 0.4
Generated Text: ... *hysteric breathing*... *hyp

In [178]:
# prompts_en_cultural | Indonesian steering (fear)
texts_generated_prompts_en_cultural_id_fear = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction_id,
    prompts=prompts_id_cultural,
    steering_vector=steering_vector_id['fear'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=350
)

In [179]:
for prompt in texts_generated_prompts_en_cultural_id_fear:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_id_fear[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: Bagaimana perasaan dan reaksi Anda jika seorang polisi menghentikan Anda, menanyai Anda, lalu mulai merekam Anda, padahal Anda tidak melakukan kesalahan apa pun?
Steering Strength: 0.2
Generated Text: Saya merasa sangat terkejut dan marah. Saya tidak pernah berduga melakukan kesalahan. Saya merasa dirugikan hak asasinya. Saya ingin mengetahui apa yang sebenarnya terjadi.
----------------------------------------
Steering Strength: 0.3
Generated Text: Bagaiman tidak merespon memang memas peran padah dan reng menghant.
----------------------------------------
Steering Strength: 0.4
Generated Text: The patient was requested the
SarreamerThe
Vocabulary
The word’s
Mention


We found the
I did the
Sitting
The table of
The system
The
Mention

Crew

The
The

A question
The
The
The
The
Mentions
The

The

The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
The
Th

In [180]:
# prompts_en_cultural | No steering baseline (fear)
texts_generated_prompts_en_cultural_no_steer_fear = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction,
    prompts=prompts_id_cultural,
    steering_vector=None,
    # steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [181]:
for prompt in texts_generated_prompts_en_cultural_no_steer_fear:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_no_steer_fear[prompt]:
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: Bagaimana perasaan dan reaksi Anda jika seorang polisi menghentikan Anda, menanyai Anda, lalu mulai merekam Anda, padahal Anda tidak melakukan kesalahan apa pun?
Generated Text: Saya merasa sangat frustrasi dan tidak nyaman. Saya merasa diserang dan dikhawatirkan, meskipun tidak melakukan kesalahan apa pun. Saya juga merasa sakit hati karena tidak ada bukti yang kuat untuk menyangkal kesalahan saya. Saya hanya ingin meminta klarifikasi dan penjelasan yang jelas tentang apa yang terjadi.
----------------------------------------
Prompt: Bagaimana perasaan dan reaksi Anda jika orang tua Anda mengetahui bahwa Anda membuat keputusan besar dalam hidup tanpa meminta pendapat mereka terlebih dahulu?
Generated Text: Saya merasa sedikit cemas dan khawatir bahwa mereka mungkin merasa kecewa atau kehilangan kepercayaan padaku. Namun, saya juga yakin bahwa saya telah berpikir dengan matang dan telah membuat keputusan yang tepat untuk diri sendiri. Saya berharap mereka akan memahami dan mend

In [ ]:
prompt = "How would you feel and react if you wanted to message a senior colleague for advice, but you were afraid they might think you were bothering them?"
generated_text = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction,
    model=model,
    steering_vector=steering_vector_eng['fear'],
    tokenizer=tokenizer,
    steering_strength=0.3,
    max_new_tokens=300,
)
print(generated_text)
generated_text = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction,
    model=model,
    steering_vector=steering_vector_id['fear'],
    tokenizer=tokenizer,
    steering_strength=0.3,
    max_new_tokens=300,
)
print(generated_text)